In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fixed seed for reproducibility.
np.random.seed(0)

n = 50
d = 16

q = np.random.randn(d)
K = np.random.randn(n, d)


def psi_star(k):
    """Return 0.5 * ||k||^2 for each input vector."""
    return 0.5 * np.sum(k**2, axis=-1)


def softmax(x):
    # Shift by the maximum for numerical stability.
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)


taus = np.logspace(-2.5, 1.0, 80)
log_tau = np.log10(taus)

# Attention weights for each temperature and key.
A = np.zeros((len(taus), n))

for i, tau in enumerate(taus):
    scores = q @ K.T - psi_star(tau * K)
    A[i] = softmax(scores)


# Sort keys by decreasing inner product with q.
base_scores = q @ K.T
order = np.argsort(-base_scores)
A = A[:, order]


# Plot log-scaled attention weights.
A_log = np.log10(A + 1e-10)

plt.figure(figsize=(6, 4))

plt.imshow(
    A_log.T,
    aspect="auto",
    origin="lower",
    extent=[log_tau[0], log_tau[-1], 0, n],
    cmap="magma",
    vmin=-6,
    vmax=0,
)

plt.xlabel(r"$\log_{10}\tau$")
plt.ylabel("key index (sorted by inner product)")
plt.title("Temperature-dependent Attention Weight")

cbar = plt.colorbar()
cbar.set_label(r"$\log_{10} a_j(\tau)$")

plt.tight_layout()

plt.savefig(
    "attention_heatmap_tau.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
